# 02 — StandardScaler

In the previous notebook, I learned how to split data into training and testing sets. Now I will learn how to scale numerical features using `StandardScaler` from Scikit-Learn.

**What I will learn:** why scaling matters, `fit()`, `transform()`, `fit_transform()`, and how to scale test data without data leakage.

## 1. Why do we need scaling?

Imagine a house dataset with two features:
- House size: 1,000 to 3,000 square feet
- Bedrooms: 1 to 5

These features have very different ranges. Some machine learning algorithms, such as KNN and models trained with gradient descent, can work better when features are on a similar scale.

`StandardScaler` uses **Z-score normalization**. It changes each feature so its training-data mean is approximately 0 and its standard deviation is approximately 1.

$$
\boxed{z = \frac{x - \mu}{\sigma}}
$$

- $x$: original value
- $\mu$: mean of the feature
- $\sigma$: standard deviation of the feature

**Note:** StandardScaler does not force values to stay between 0 and 1.

## 2. Create a small dataset

Each row represents one house. The first column is house size and the second is the number of bedrooms.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

X = np.array([
    [1000, 2],
    [1500, 3],
    [2000, 4],
    [2500, 5]
])

print("Original data:\n", X)
print("Shape:", X.shape)

## 3. Create a scaler and use `fit_transform()`

`fit_transform()` does two things:
1. **Fit:** learn the mean and standard deviation of each column.
2. **Transform:** use those values to scale the data.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled data:\n", np.round(X_scaled, 2))
print("Mean of original columns:", scaler.mean_)
print("Standard deviation used:", scaler.scale_)

# here scaler is the object of class StanderdScaler()
# class StanderdScaler() have method fit_transform() which first calculate the mean, sd of data then z-score and all the data points transforms
 # corresponding to the original data points
 # now both features are in similar scale 

The scaled values can be negative. A negative value simply means it is below that feature's mean.

## 4. `fit()` vs `transform()` vs `fit_transform()`

| Method | Meaning |
|---|---|
| `fit(X)` | Learn the mean and standard deviation from `X`. |
| `transform(X)` | Scale `X` using values already learned. |
| `fit_transform(X)` | Learn and scale in one step. |

These two approaches give the same result when used on the same data.

In [ ]:
scaler_a = StandardScaler()
scaler_a.fit(X)
result_a = scaler_a.transform(X)

scaler_b = StandardScaler()
result_b = scaler_b.fit_transform(X)

print("Same result:", np.allclose(result_a, result_b))

## 5. Split first, then scale

This is the most important part. We should **not** fit the scaler on the whole dataset before splitting it. That would allow information from the test set to influence preprocessing. This is called **data leakage**.

The correct order is:
1. Split into training and testing sets.
2. Fit the scaler **only on training features**.
3. Transform training features.
4. Transform testing features using the **same scaler**.

In [ ]:
from sklearn.model_selection import train_test_split

X_houses = np.array([
    [1000, 2], [1200, 2], [1400, 3], [1600, 3],
    [1800, 3], [2000, 4], [2200, 4], [2400, 4],
    [2600, 5], [2800, 5]
])

y_prices = np.array([150, 170, 190, 210, 230, 250, 270, 290, 310, 330])

X_train, X_test, y_train, y_test = train_test_split(
    X_houses, y_prices, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Scaled training data:\n", np.round(X_train_scaled, 2))
print("Scaled testing data:\n", np.round(X_test_scaled, 2))

**Why not use `fit_transform(X_test)`?** Because it would calculate a new mean and standard deviation from the test set. During testing, we must use the scaling rules learned from training data.

Also, the test set's scaled mean does **not** have to be 0. That is normal.

## 6. Train a simple model using scaled features

Let's connect this notebook with what I learned about `fit()`, `predict()`, and `score()`. Here we use `SGDRegressor`, which learns using gradient descent.

In [ ]:
from sklearn.linear_model import SGDRegressor

model = SGDRegressor(max_iter=10000, tol=1e-4, random_state=42)
model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)

print("Actual prices:", y_test)
print("Predicted prices:", np.round(predictions, 2))
print("Test R²:", round(model.score(X_test_scaled, y_test), 3))

This is only a tiny practice dataset, so its test R² is not a reliable measure of real-world performance. The goal here is to understand the preprocessing workflow.

## 7. `inverse_transform()`

Sometimes I want to convert scaled features back to their original units. `inverse_transform()` does that.

In [ ]:
X_original_again = scaler.inverse_transform(X_test_scaled)

print("Scaled test data:\n", np.round(X_test_scaled, 2))
print("Back to original values:\n", X_original_again)

## 8. Practice tasks

1. Change one house size in `X_houses`. Run the split and scaling again. What changes?
2. Print `scaler.mean_`, `scaler.var_`, and `scaler.scale_`. What does each attribute contain?
3. Compare `X_test` with `X_test_scaled`. Are the number of rows and columns the same?
4. Try `scaler.inverse_transform(X_train_scaled)`. Do you get your original training features back?
5. In your own words, explain why we use `transform(X_test)` instead of `fit_transform(X_test)`.

## 9. Quick notes for revision

```python
# StandardScaler uses z = (x - mean) / standard_deviation.
# fit() learns mean and standard deviation from the given data.
# transform() scales data using the already learned values.
# fit_transform() does both steps together.
# Split first. Fit the scaler only on X_train.
# Use the same scaler to transform X_train and X_test.
# inverse_transform() converts scaled features back to original units.
```

**My main takeaway:** I should learn scaling rules from training data and apply the same rules to unseen data.